# P(P)N2V Convalaria demo

In [ ]:
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tifffile

from careamics import NoiseModelTrainer
from careamics.config import create_advanced_n2v_config, create_advanced_pn2v_config
from careamics.config.noise_model import GaussianMixtureNMConfig
from careamics.lightning import (
    CareamicsDataModule,
    ConfigSaverCallback,
    N2VModule,
    PN2VModule,
    convert_prediction,
)
from careamics.models.lvae.noise_models import GaussianMixtureNoiseModel
from careamics.metrics.metrics import psnr, scale_invariant_psnr


In [ ]:
# Set seeds for reproducibility
from lightning.pytorch import seed_everything

seed = 42
seed_everything(seed)

### Load data and set paths to it

In [ ]:
root_path = Path(".")
data_path = root_path / "Convallaria_diaphragm"
if not data_path.exists():
    zip_path = root_path / "Convallaria_diaphragm.zip"
    urllib.request.urlretrieve(
        "https://zenodo.org/record/5156913/files/Convallaria_diaphragm.zip?download=1",
        zip_path,
    )
    with zipfile.ZipFile(zip_path) as zip_file:
        zip_file.extractall(root_path)

noisy = tifffile.imread(
    data_path / "20190520_tl_25um_50msec_05pc_488_130EM_Conv.tif"
).astype(np.float32)
calibration = tifffile.imread(
    data_path / "20190726_tl_50um_500msec_wf_130EM_FD.tif"
).astype(np.float32)

train_data = noisy[:-5]
val_data = noisy[-5:, :192, :192]
test_data = noisy[:, :512, :512]
gt = test_data.mean(axis=0)

normalization_params = {
    "input_means": [float(noisy.mean())],
    "input_stds": [float(noisy.std())],
}

### Visualize a single noisy and calibration image

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(noisy[0], cmap="gray")
ax[0].set_title("Noisy Image")
ax[1].imshow(calibration[0], cmap="gray")
ax[1].set_title("Calibration Image")

### Train the calibration noise model

In [ ]:
calibration_signal = np.repeat(
    calibration.mean(axis=0, keepdims=True), calibration.shape[0], axis=0
)
noise_model_trainer = NoiseModelTrainer(n_gaussian=3, n_coeff=2, min_sigma=50.0)
noise_model_trainer.train_from_pairs(
    signal=calibration_signal, observation=calibration, axes="SYX"
)
calibration_noise_model = noise_model_trainer.get_config().noise_models[0]

### Create config

In [ ]:
config = create_advanced_pn2v_config(
    experiment_name="convallaria_pn2v_calibration",
    data_type="array",
    axes="SYX",
    patch_size=(96, 96),
    batch_size=5,
    noise_model=calibration_noise_model,
    num_epochs=200,
    num_steps=20,
    num_classes=800,
    roi_size=5,
    masked_pixel_percentage=3.125,
    normalization_params=normalization_params,
    model_params={"depth": 3, "num_channels_init": 64, "use_batch_norm": False},
    trainer_params={"accumulate_grad_batches": 4},
    optimizer_params={"lr": 1e-3},
    lr_scheduler_params={"factor": 0.5, "patience": 10},
    num_workers=0,
    seed=seed,
)

### Create Lightning datamodule and model

In [ ]:
train_data_module = CareamicsDataModule(
    data_config=config.data_config,
    train_data=train_data,
    val_data=val_data,
)

model = PN2VModule(config.algorithm_config)

### Manually initialize the datamodule and visualize single train and val batches

In [ ]:
train_data_module.setup("fit")
train_data_module.setup("validate")

train_batch = next(iter(train_data_module.train_dataloader()))
val_batch = next(iter(train_data_module.val_dataloader()))

fig, ax = plt.subplots(1, 5, figsize=(10, 5))
ax[0].set_title("Training Batch")
for i in range(5):
    ax[i].imshow(train_batch[0].data[i][0].numpy(), cmap="gray")

fig, ax = plt.subplots(1, 5, figsize=(10, 5))
ax[0].set_title("Validation Batch")
for i in range(5):
    ax[i].imshow(val_batch[0].data[i][0].numpy(), cmap="gray")

### Train the model

In [ ]:
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import ModelCheckpoint

root = Path("convallaria_pn2v")
callbacks = [
    ModelCheckpoint(
        dirpath=root / "checkpoints",
        filename=config.experiment_name,
        save_last=True,
        monitor="val_loss",
        mode="min",
    ),
    ConfigSaverCallback(
        config.version,
        config.experiment_name,
        config.training_config,
        config.data_config,
    ),
]

trainer = Trainer(
    default_root_dir=root,
    callbacks=callbacks,
    **config.training_config.trainer_params,
)
trainer.fit(model, datamodule=train_data_module)

### Create an inference config and datamodule

In [ ]:
inf_data_config = config.data_config.convert_mode(
    new_mode="predicting",
    new_patch_size=(192, 192),
    overlap_size=(48, 48),
    new_batch_size=1,
)

inf_data_module = CareamicsDataModule(data_config=inf_data_config, pred_data=test_data)

### Predict and stitch the tiles

In [ ]:
predictions = trainer.predict(model, datamodule=inf_data_module)
predictions, _ = convert_prediction(predictions, tiled=True, restore_shape=True)
prediction = predictions[0].squeeze()

### Visualize predictions and count metrics

In [ ]:
data_range = gt.max() - gt.min()

images = [0, 50, 99]
fig, ax = plt.subplots(3, 3, figsize=(15, 15))
fig.tight_layout()

for i in range(3):
    psnr_noisy = psnr(gt, test_data[images[i]], data_range=data_range)
    psnr_result = psnr(gt, prediction[images[i]], data_range=data_range)
    scale_invariant_psnr_result = scale_invariant_psnr(gt, prediction[images[i]])

    ax[i, 0].imshow(test_data[images[i]], cmap="gray")
    ax[i, 0].title.set_text(f"Noisy\nPSNR: {psnr_noisy:.2f}")

    ax[i, 1].imshow(prediction[images[i]], cmap="gray")
    ax[i, 1].title.set_text(
        f"Prediction\nPSNR: {psnr_result:.2f}\n"
        f"Scale invariant PSNR: {scale_invariant_psnr_result:.2f}"
    )

    ax[i, 2].imshow(gt, cmap="gray")
    ax[i, 2].title.set_text("Ground-truth")

In [ ]:
psnrs = np.array([psnr(gt, frame, data_range=data_range) for frame in prediction])
scale_invariant_psnrs = np.array([scale_invariant_psnr(gt, frame) for frame in prediction])

print(f"PSNR: {psnrs.mean():.2f} +/- {psnrs.std():.2f}")
print(
    f"Scale invariant PSNR: "
    f"{scale_invariant_psnrs.mean():.2f} +/- {scale_invariant_psnrs.std():.2f}"
)
print("Reported PSNR: 36.47")

### Train N2V for the bootstrap noise model

In [ ]:
n2v_config = create_advanced_n2v_config(
    experiment_name="convallaria_n2v",
    data_type="array",
    axes="SYX",
    patch_size=(96, 96),
    batch_size=20,
    num_epochs=200,
    num_steps=10,
    roi_size=5,
    masked_pixel_percentage=3.125,
    normalization_params=normalization_params,
    model_params={"depth": 3, "num_channels_init": 64, "use_batch_norm": False},
    optimizer_params={"lr": 1e-3},
    lr_scheduler_params={"factor": 0.5, "patience": 10},
    num_workers=0,
    seed=seed,
)

n2v_data_module = CareamicsDataModule(
    data_config=n2v_config.data_config,
    train_data=train_data,
    val_data=val_data,
)
n2v_model = N2VModule(n2v_config.algorithm_config)

n2v_trainer = Trainer(
    default_root_dir=root,
    callbacks=[
        ModelCheckpoint(
            dirpath=root / "checkpoints",
            filename=n2v_config.experiment_name,
            save_last=True,
            monitor="val_loss",
            mode="min",
        ),
        ConfigSaverCallback(
            n2v_config.version,
            n2v_config.experiment_name,
            n2v_config.training_config,
            n2v_config.data_config,
        ),
    ],
    **n2v_config.training_config.trainer_params,
)
n2v_trainer.fit(n2v_model, datamodule=n2v_data_module)

### Predict the noisy movie with N2V

In [ ]:
n2v_inf_data_config = n2v_config.data_config.convert_mode(
    new_mode="predicting",
    new_patch_size=(256, 256),
    overlap_size=(48, 48),
    new_batch_size=8,
)
n2v_inf_data_module = CareamicsDataModule(
    data_config=n2v_inf_data_config, pred_data=noisy
)

n2v_predictions = n2v_trainer.predict(n2v_model, datamodule=n2v_inf_data_module)
n2v_predictions, _ = convert_prediction(
    n2v_predictions, tiled=True, restore_shape=True
)
n2v_prediction = n2v_predictions[0].squeeze()

### Train the bootstrap noise model

In [ ]:
bootstrap_noise_model_config = GaussianMixtureNMConfig(
    min_signal=float(np.percentile(n2v_prediction, 0.5)),
    max_signal=float(np.percentile(n2v_prediction, 99.5)),
    n_gaussian=3,
    n_coeff=2,
    min_sigma=50.0,
)
bootstrap_gmm = GaussianMixtureNoiseModel(bootstrap_noise_model_config)
bootstrap_gmm.fit(
    signal=n2v_prediction, observation=noisy, lower_clip=0.5, upper_clip=99.5
)
bootstrap_noise_model = bootstrap_noise_model_config.model_copy(
    update={"weight": bootstrap_gmm.weight.detach().cpu().numpy()}
)

### Train PN2V with the bootstrap noise model

In [ ]:
bootstrap_config = create_advanced_pn2v_config(
    experiment_name="convallaria_pn2v_bootstrap",
    data_type="array",
    axes="SYX",
    patch_size=(96, 96),
    batch_size=5,
    noise_model=bootstrap_noise_model,
    num_epochs=200,
    num_steps=20,
    num_classes=800,
    roi_size=5,
    masked_pixel_percentage=3.125,
    normalization_params=normalization_params,
    model_params={"depth": 3, "num_channels_init": 64, "use_batch_norm": False},
    trainer_params={"accumulate_grad_batches": 4},
    optimizer_params={"lr": 1e-3},
    lr_scheduler_params={"factor": 0.5, "patience": 10},
    num_workers=0,
    seed=seed,
)

bootstrap_data_module = CareamicsDataModule(
    data_config=bootstrap_config.data_config,
    train_data=train_data,
    val_data=val_data,
)
bootstrap_model = PN2VModule(bootstrap_config.algorithm_config)

bootstrap_trainer = Trainer(
    default_root_dir=root,
    callbacks=[
        ModelCheckpoint(
            dirpath=root / "checkpoints",
            filename=bootstrap_config.experiment_name,
            save_last=True,
            monitor="val_loss",
            mode="min",
        ),
        ConfigSaverCallback(
            bootstrap_config.version,
            bootstrap_config.experiment_name,
            bootstrap_config.training_config,
            bootstrap_config.data_config,
        ),
    ],
    **bootstrap_config.training_config.trainer_params,
)
bootstrap_trainer.fit(bootstrap_model, datamodule=bootstrap_data_module)

### Predict and count metrics

In [ ]:
bootstrap_inf_data_config = bootstrap_config.data_config.convert_mode(
    new_mode="predicting",
    new_patch_size=(192, 192),
    overlap_size=(48, 48),
    new_batch_size=1,
)
bootstrap_inf_data_module = CareamicsDataModule(
    data_config=bootstrap_inf_data_config, pred_data=test_data
)

bootstrap_predictions = bootstrap_trainer.predict(
    bootstrap_model, datamodule=bootstrap_inf_data_module
)
bootstrap_predictions, _ = convert_prediction(
    bootstrap_predictions, tiled=True, restore_shape=True
)
bootstrap_prediction = bootstrap_predictions[0].squeeze()

fig, ax = plt.subplots(3, 3, figsize=(15, 15))
fig.tight_layout()

for i in range(3):
    psnr_noisy = psnr(gt, test_data[images[i]], data_range=data_range)
    psnr_result = psnr(gt, bootstrap_prediction[images[i]], data_range=data_range)
    scale_invariant_psnr_result = scale_invariant_psnr(
        gt, bootstrap_prediction[images[i]]
    )

    ax[i, 0].imshow(test_data[images[i]], cmap="gray")
    ax[i, 0].title.set_text(f"Noisy\nPSNR: {psnr_noisy:.2f}")

    ax[i, 1].imshow(bootstrap_prediction[images[i]], cmap="gray")
    ax[i, 1].title.set_text(
        f"Prediction\nPSNR: {psnr_result:.2f}\n"
        f"Scale invariant PSNR: {scale_invariant_psnr_result:.2f}"
    )

    ax[i, 2].imshow(gt, cmap="gray")
    ax[i, 2].title.set_text("Ground-truth")

In [ ]:
psnrs = np.array(
    [psnr(gt, frame, data_range=data_range) for frame in bootstrap_prediction]
)
scale_invariant_psnrs = np.array(
    [scale_invariant_psnr(gt, frame) for frame in bootstrap_prediction]
)

print(f"PSNR: {psnrs.mean():.2f} +/- {psnrs.std():.2f}")
print(
    f"Scale invariant PSNR: "
    f"{scale_invariant_psnrs.mean():.2f} +/- {scale_invariant_psnrs.std():.2f}"
)
print("Reported PSNR: 36.70")